## Prediction of Defective Units Rate

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import pickle

import warnings
warnings.filterwarnings('ignore')

In [2]:
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import VotingRegressor, RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import SGDRegressor

In [3]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [4]:
import joblib
from sklearn.pipeline import Pipeline

In [5]:
df = pd.read_csv('Procurement KPI Data.csv')

In [6]:
df.head()

,PO_ID,Supplier,Item_Category,Quantity,Defective_Units,Compliance,Lead_Time,Negotiated_vs_Unit_Price,Price_Ratio
0,PO-00001,Alpha_Inc,Office Supplies,1176,NaN,Yes,8.0,2.32,0.884749
1,PO-00002,Delta_Logistics,Office Supplies,1509,235.0,Yes,10.0,1.98,0.949644
2,PO-00003,Gamma_Co,MRO,910,41.0,Yes,20.0,3.25,0.965972
3,PO-00004,Beta_Supplies,Packaging,1344,112.0,Yes,19.0,4.33,0.956635
4,PO-00005,Delta_Logistics,Raw Materials,1180,171.0,No,12.0,3.54,0.944748


In [7]:
df.columns

Index(['PO_ID', 'Supplier', 'Item_Category', 'Quantity', 'Defective_Units',
       'Compliance', 'Lead_Time', 'Negotiated_vs_Unit_Price', 'Price_Ratio'],
      dtype='object')

## Train-Test Split

In [8]:
X = df[['Supplier', 'Item_Category', 'Quantity',
       'Compliance', 'Lead_Time', 'Negotiated_vs_Unit_Price']]

y = df['Defective_Units']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
df.head()

,PO_ID,Supplier,Item_Category,Quantity,Defective_Units,Compliance,Lead_Time,Negotiated_vs_Unit_Price,Price_Ratio
0,PO-00001,Alpha_Inc,Office Supplies,1176,NaN,Yes,8.0,2.32,0.884749
1,PO-00002,Delta_Logistics,Office Supplies,1509,235.0,Yes,10.0,1.98,0.949644
2,PO-00003,Gamma_Co,MRO,910,41.0,Yes,20.0,3.25,0.965972
3,PO-00004,Beta_Supplies,Packaging,1344,112.0,Yes,19.0,4.33,0.956635
4,PO-00005,Delta_Logistics,Raw Materials,1180,171.0,No,12.0,3.54,0.944748


In [10]:
X_train.head()

,Supplier,Item_Category,Quantity,Compliance,Lead_Time,Negotiated_vs_Unit_Price
739,Delta_Logistics,MRO,241,Yes,7.0,5.27
133,Beta_Supplies,Office Supplies,1912,Yes,10.0,0.91
234,Delta_Logistics,Raw Materials,239,Yes,NaN,4.40
55,Epsilon_Group,Raw Materials,881,Yes,2.0,5.89
639,Beta_Supplies,MRO,502,Yes,15.0,9.51


In [28]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 156 entries, 568 to 292
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Supplier                  156 non-null    object 
 1   Item_Category             156 non-null    object 
 2   Quantity                  156 non-null    float64
 3   Compliance                156 non-null    object 
 4   Lead_Time                 156 non-null    float64
 5   Negotiated_vs_Unit_Price  156 non-null    float64
dtypes: float64(3), object(3)
memory usage: 8.5+ KB


In [12]:
df.isnull().sum()

PO_ID                         0
Supplier                      0
Item_Category                 0
Quantity                      0
Defective_Units             136
Compliance                    0
Lead_Time                    87
Negotiated_vs_Unit_Price      0
Price_Ratio                   0
dtype: int64

## Load Feature Engineering Pipeline

In [13]:
def lead_time_fill(df):

    df['Lead_Time'] = df['Lead_Time'].fillna(df['Lead_Time'].mean())

    return df

In [14]:
def outliers_capping(df):

   for col in ['Quantity','Defective_Units']:
      
      q3 = df[col].quantile(0.75)
      q1 = df[col].quantile(0.25)
      iqr = q3 - q1
      lb = q1 - 1.5*iqr
      ub = q3 + 1.5*iqr

      df[col] = df[col].clip(lower=lb, upper=ub)

      return df

In [15]:
with open('feature_engineering_pipeline.pkl', 'rb') as f:
    preprocessing_loaded = pickle.load(f)

In [16]:
preprocessing_loaded

Pipeline(steps=[('functiontransformer-1',
                 FunctionTransformer(func=<function lead_time_fill at 0x000002D740766660>)),
                ('functiontransformer-2',
                 FunctionTransformer(func=<function outliers_capping at 0x000002D740766520>)),
                ('columntransformer',
                 ColumnTransformer(transformers=[('numerical_trf',
                                                  Pipeline(steps=[('transforming',
                                                                   PowerTransformer()),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Quantity',
                                                   'Negotiated_vs_Unit_Price',
                                                   'Lead_Time']),
                                                 ('encoding_cat_trf',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse=False))]),
                                                  ['Supplier', 'Item_Category',
                                                   'Compliance'])]))])

## Apply Feature Engineering


In [17]:
X_train_trf = preprocessing_loaded.fit_transform(X_train)
X_test_trf = preprocessing_loaded.transform(X_test)

In [18]:
y_train_reshaped = y_train.values.reshape(-1, 1)
y_test_reshaped = y_test.values.reshape(-1, 1)

In [19]:
imputer = SimpleImputer(strategy='median')
y_train_trf = imputer.fit_transform(y_train_reshaped)
y_test_trf = imputer.transform(y_test_reshaped)

yeo = PowerTransformer(method='yeo-johnson')
y_train_yeo = yeo.fit_transform(y_train_trf)
y_test_yeo = yeo.transform(y_test_trf)

scaler_y = StandardScaler()
y_train_trf = scaler_y.fit_transform(y_train_yeo).ravel()
y_test_trf = scaler_y.transform(y_test_yeo).ravel()

## Model Training & Evaluation

In [20]:
models = {
    'Linear Regression': LinearRegression(),
    'Lasso': Lasso(alpha=0.1),
    'Ridge': Ridge(alpha=0.1),
    'Batch GD': SGDRegressor(max_iter=1000),
    'DecisionTreeRegressor': DecisionTreeRegressor(max_depth=5)
}

results = {}

n = X_test_trf.shape[0]     # number of samples
p = X_test_trf.shape[1]     # number of features

for name, model in models.items():
    model.fit(X_train_trf, y_train_trf)
    y_pred = model.predict(X_test_trf)

    mse = mean_squared_error(y_test_trf, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_trf, y_pred)
    r2 = r2_score(y_test_trf, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    results[name] = {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Adj R2": adj_r2
    }

results_df = pd.DataFrame(results).T
print("Model Performance Metrics:\n")
print(results_df)

best_model = results_df["R2"].idxmax()
print(f"\nBest model based on R²: {best_model}")

Model Performance Metrics:

                            MSE      RMSE       MAE        R2    Adj R2
Linear Regression      0.360566  0.600472  0.434016  0.692576  0.659638
Lasso                  0.620019  0.787413  0.669797  0.471363  0.414723
Ridge                  0.360624  0.600520  0.434180  0.692526  0.659583
Batch GD               0.373724  0.611329  0.454605  0.681358  0.647217
DecisionTreeRegressor  0.391158  0.625426  0.416972  0.666493  0.630760

Best model based on R²: Linear Regression


## Hyperparameter Tuning Using GridSearchCV

In [21]:
params = {
    'alpha':[0.01,0.05,0.07,0.1,10]
   }

grid = GridSearchCV(estimator=Ridge(), param_grid=params, cv=5, scoring='r2', n_jobs=-1)

grid.fit(X_train_trf, y_train_trf)

print("Best Parameters:", grid.best_params_)
print("Best CV R2:", grid.best_score_)

Best Parameters: {'alpha': 0.1}
Best CV R2: 0.6851853526108016


## Save Model

In [22]:
best_ridge = grid.best_estimator_
best_ridge

Ridge(alpha=0.1)

In [24]:
joblib.dump(best_ridge, 'model.pkl')

['model.pkl']

## Prediction

In [26]:
model = joblib.load('model.pkl')

y_pred = model.predict(X_test_trf)
y_pred

array([ 0.39552147,  1.43330496,  1.13923708,  1.31202362, -0.08428652,
        0.00399981,  0.72507787, -1.91586119, -0.70095349, -0.08181044,
       -1.38299955,  0.85654154, -1.95038759, -0.06075991, -0.02737133,
        0.39094235,  0.3799357 , -1.1826443 , -1.04535634, -0.2220042 ,
        1.73410547, -0.41895334, -0.4802405 , -0.62596347,  0.00291083,
       -0.97114086, -0.19589602, -0.07134134,  1.32282279,  1.3178615 ,
        1.05365463, -0.73755865,  0.70454508,  1.53635786, -0.99993264,
       -1.02561037,  0.33228476, -0.48627814,  0.26954542, -1.77592369,
       -0.05690976,  1.30350485,  0.62421879, -0.23445499, -0.86004963,
       -0.97909322,  0.27654854, -0.02800736, -1.50057894,  0.0559745 ,
        0.33141855, -0.52128114,  0.21816667,  1.04415518, -0.8125586 ,
        0.96990047,  0.32285683, -0.01232211, -0.88250463,  1.62495717,
       -0.48943925, -0.50801127, -0.95550697, -0.53360532, -0.22524294,
        1.61900641, -1.44019968,  1.55791053,  0.02271558,  0.40